In [6]:
import pandas as pd
import numpy as np
from datetime import datetime, timedelta

# --- ПАРАМЕТРЫ ГЕНЕРАЦИИ ---
n_rows = 5000
np.random.seed(42)
today = datetime(2026, 7, 7)

# Справочники
product_types = {1: "Мопед 1", 2: "Мопед 2", 3: "Мопед 3"}
parts = {
    101: "Двигатель",
    102: "Рама",
    103: "Колесо",
    104: "Руль",
    105: "Фара",
    106: "Сиденье",
    107: "Топливный бак",
    108: "Тормозная система",
    109: "Цепь",
    110: "Аккумулятор"
}

# Детали, которые систематически отстают от сроков
problematic_parts = [101, 102]

# Детали с высоким процентом брака
high_scrap_parts = [108, 110]

# Проблемные месяцы (март и август 2026)
problematic_months = [3, 8]

# --- ГЕНЕРАЦИЯ БАЗОВЫХ ДАННЫХ ---
data = {
    "order_id": range(1, n_rows + 1),
    "product_type_id": np.random.choice(list(product_types.keys()), n_rows),
    "part_id": np.random.choice(list(parts.keys()), n_rows),
    "workstation_id": np.random.randint(1, 11, n_rows),
    "planned_quantity": np.random.randint(10, 101, n_rows)
}

df = pd.DataFrame(data)
df["product_type_name"] = df["product_type_id"].map(product_types)
df["part_name"] = df["part_id"].map(parts)

# Генерируем плановые даты в пределах 2026 года
start_date_base = datetime(2026, 2, 1)
planned_end_dates = [start_date_base + timedelta(days=int(x)) for x in np.random.uniform(0, 333, n_rows)]
df["planned_end_date"] = planned_end_dates

planned_durations = np.random.randint(7, 31, n_rows)
df["planned_start_date"] = df["planned_end_date"] - pd.to_timedelta(planned_durations, unit='D')

# --- ГЕНЕРАЦИЯ ФАКТИЧЕСКИХ ДАННЫХ И ОБЪЕМОВ ---
actual_end_dates = []
produced_quantities = []
accepted_quantities = []

is_problematic = df["part_id"].isin(problematic_parts)
is_high_scrap = df["part_id"].isin(high_scrap_parts)

for i in range(n_rows):
    planned_end = df.loc[i, "planned_end_date"]
    planned_qty = df.loc[i, "planned_quantity"]
    is_prob = is_problematic.iloc[i]
    is_h_scrap = is_high_scrap.iloc[i]
    planned_month = planned_end.month
    
    # Определяем временную зону
    if planned_end < today - timedelta(days=7):
        not_delivered = np.random.rand() < 0.1
    elif planned_end < today:
        not_delivered = np.random.rand() < 0.1
    else:
        not_delivered = np.random.rand() < 0.1
    
    if not_delivered:
        actual_end = pd.NaT
        produced_qty = np.random.randint(0, planned_qty)
        
        # Расчет брака
        if is_h_scrap:
            # Высокий брак для проблемных деталей (15-30%)
            scrap_rate = np.random.uniform(0.15, 0.30)
        elif planned_month in problematic_months:
            # Высокий брак в проблемные месяцы (12-25%)
            scrap_rate = np.random.uniform(0.12, 0.25)
        else:
            # Обычный брак (0-10%)
            scrap_rate = np.random.uniform(0.0, 0.10)
        
        accepted_qty = int(produced_qty * (1 - scrap_rate))
    else:
        # Заказ сдан
        if is_prob:
            # Систематическая просрочка для проблемных деталей
            if planned_month in problematic_months:
                # В проблемные месяцы еще большая задержка
                delay = np.random.randint(15, 40)
            else:
                delay = np.random.randint(10, 31)
        else:
            r = np.random.rand()
            if planned_month in problematic_months:
                # В проблемные месяцы больше просрочек
                if r < 0.3:
                    delay = np.random.randint(-5, 0)
                elif r < 0.5:
                    delay = 0
                else:
                    delay = np.random.randint(1, 15)
            else:
                if r < 0.5:
                    delay = np.random.randint(-5, 0)
                elif r < 0.8:
                    delay = 0
                else:
                    delay = np.random.randint(1, 8)
        
        actual_end = planned_end + timedelta(days=delay)
        
        # Расчет брака
        if is_h_scrap:
            scrap_rate = np.random.uniform(0.15, 0.30)
        elif planned_month in problematic_months:
            scrap_rate = np.random.uniform(0.12, 0.25)
        else:
            scrap_rate = np.random.uniform(0.0, 0.10)
        
        produced_qty = planned_qty
        accepted_qty = int(produced_qty * (1 - scrap_rate))
    
    actual_end_dates.append(actual_end)
    produced_quantities.append(produced_qty)
    accepted_quantities.append(accepted_qty)

df["actual_end_date"] = actual_end_dates
df["produced_quantity"] = produced_quantities
df["accepted_quantity"] = accepted_quantities

# --- ГЕНЕРАЦИЯ СТАТУСОВ ЗАКАЗОВ ---
def determine_status(row):
    if pd.isna(row["actual_end_date"]):
        if row["planned_end_date"] < today - timedelta(days=7):
            return "В зоне риска"
        else:
            return "В производстве"
    else:
        if row["actual_end_date"] > row["planned_end_date"]:
            return "Просрочен"
        else:
            return "Завершен"

df["order_status"] = df.apply(determine_status, axis=1)

# Задаем финальный порядок колонок
cols = [
    "order_id", "product_type_id", "product_type_name", 
    "part_id", "part_name", "workstation_id", 
    "planned_start_date", "planned_end_date", "planned_quantity", 
    "actual_end_date", "produced_quantity", "accepted_quantity",
    "order_status"
]
df = df[cols]

# --- СОХРАНЕНИЕ В XLSX ---
output_file = "moped_production_dataset.xlsx"

with pd.ExcelWriter(output_file, engine='xlsxwriter', datetime_format='yyyy-mm-dd') as writer:
    df.to_excel(writer, index=False, sheet_name='Production Data')
    
    workbook = writer.book
    worksheet = writer.sheets['Production Data']
    for i, col in enumerate(df.columns):
        max_len = max(df[col].astype(str).map(len).max(), len(col)) + 2
        worksheet.set_column(i, i, max_len)

print(f"✅ Датасет на {n_rows} строк успешно сгенерирован и сохранен в файл: {output_file}")
print(f"\n📊 Распределение статусов:")
print(df["order_status"].value_counts())
print(f"\n🔧 Статистика по деталям с высоким браком:")
for part_id in high_scrap_parts:
    part_data = df[df["part_id"] == part_id]
    avg_scrap = ((part_data["produced_quantity"] - part_data["accepted_quantity"]) / part_data["produced_quantity"]).mean() * 100
    print(f"{parts[part_id]}: средний брак {avg_scrap:.1f}%")
print(f"\n📅 Статистика по проблемным месяцам:")
for month in problematic_months:
    month_data = df[df["planned_end_date"].dt.month == month]
    overdue_pct = (month_data["order_status"] == "Просрочен").mean() * 100
    avg_scrap = ((month_data["produced_quantity"] - month_data["accepted_quantity"]) / month_data["produced_quantity"].replace(0, np.nan)).mean() * 100
    print(f"Месяц {month}: просрочек {overdue_pct:.1f}%, средний брак {avg_scrap:.1f}%")

✅ Датасет на 5000 строк успешно сгенерирован и сохранен в файл: moped_production_dataset.xlsx

📊 Распределение статусов:
order_status
Завершен          2665
Просрочен         1800
В производстве     296
В зоне риска       239
Name: count, dtype: int64

🔧 Статистика по деталям с высоким браком:
Тормозная система: средний брак 23.8%
Аккумулятор: средний брак 24.0%

📅 Статистика по проблемным месяцам:
Месяц 3: просрочек 53.8%, средний брак 21.6%
Месяц 8: просрочек 52.6%, средний брак 21.2%
